### Complex Numbers

A complex number $z = a + bi$ has a real part $a$ and an imaginary part $b$,
where $i = \sqrt{-1}$.

Complex numbers are the language the rest of this topic is written in. A Fourier
transform represents a signal as a sum of rotating vectors, and a single complex
number is exactly what encodes one such rotation: its **modulus** gives the size
of the vector and its **argument** gives the angle. Everything below is
groundwork for reading that.

Python uses `j` for the imaginary unit (engineering convention) but we
display results using the standard mathematical $i$. The `fmt` helper in the
first cell does that conversion, so every printed result reads the way it would
be written by hand.

### Operations covered here

- **Arithmetic** ($+$, $-$, $\times$, $\div$)
- **Exponentiation**, $z^n$
- The **complex conjugate** $\bar{z} = a - bi$
- The **modulus** $|z| = \sqrt{a^2+b^2}$
- The **argument** $\arg z = \theta$, the angle $z$ makes in the complex plane

The three numbers defined in the first cell are chosen to exercise different
cases: $z_1 = -5.9 - 7.5i$ sits in the third quadrant, $z_2 = \sqrt{2} + \pi i$
has irrational parts, and $z_3 = 7 - 2i$ is written with Python's native `j`
literal rather than the `complex()` constructor.

In [ ]:
"""complex_numbers.ipynb"""

# Cell 01 - Define three complex numbers

import numpy as np
from IPython.display import Math, display


def fmt(z: complex, places: int = 4) -> str:
    """Format a complex number as a+bi LaTeX string (uses i not j)."""
    r = round(z.real, places)
    i = round(z.imag, places)
    sign = "+" if i >= 0 else "-"
    return rf"{r}{sign}{abs(i)}i"


z1 = complex(-5.9, -7.5)
z2 = complex(np.sqrt(2), np.pi)
z3 = 7 - 2j

display(Math(rf"z_1 = {fmt(z1)}"))
display(Math(rf"z_2 = {fmt(z2)}"))
display(Math(rf"z_3 = {fmt(z3)}"))

---
### Arithmetic

Complex arithmetic follows the standard rules with $i^2 = -1$:

$$
(a+bi)+(c+di) = (a+c)+(b+d)i
\qquad
(a+bi)(c+di) = (ac-bd)+(ad+bc)i
$$

Addition and subtraction work component by component, exactly like adding
vectors in the plane. Multiplication does not: the $-bd$ term comes from
$i^2 = -1$, and it is what makes multiplication a *rotation* rather than a
stretch. Multiplying two complex numbers multiplies their moduli and adds their
arguments, which is the single most useful fact in this notebook.

Division multiplies numerator and denominator by the conjugate of the denominator
to eliminate the imaginary part from the denominator.

Python implements all four operators natively on the `complex` type, so the cell
below is just the ordinary `+ - * /` applied to $z_1$ and $z_2$.

In [ ]:
# Cell 02 - Arithmetic operations on z1 and z2

display(Math(rf"z_1 + z_2 = {fmt(z1 + z2)}"))
display(Math(rf"z_1 - z_2 = {fmt(z1 - z2)}"))
display(Math(rf"z_1 \times z_2 = {fmt(z1 * z2)}"))
display(Math(rf"\frac{{z_1}}{{z_2}} = {fmt(z1 / z2)}"))

---
### Exponentiation

Complex exponentiation follows the same `**` operator as real numbers.
Python's standard power operator works directly on complex types.

In polar form the result is easy to predict: raising $z$ to the $n$th power
raises the modulus to the $n$th power and multiplies the argument by $n$,

$$z^n = |z|^n e^{\,i n \theta}.$$

For $z_1$, whose modulus is about $9.54$, cubing gives a modulus of roughly
$9.54^3 \approx 869$, which is why $z_1^3$ below has such large components
compared with $z_1$ itself.

In [ ]:
# Cell 03 - z1 cubed

display(Math(rf"z_1^3 = {fmt(z1**3)}"))

---
### The complex conjugate

The conjugate $\bar{z} = a - bi$ reflects $z$ across the real axis.

The product $z\bar{z} = a^2 + b^2 = |z|^2$ is always real and non-negative,
which is why the conjugate appears in the denominator of complex division:
multiplying top and bottom by $\bar{z}$ turns the denominator into a plain real
number that can be divided through.

Geometrically, conjugation negates the argument while leaving the modulus alone,
so $\bar{z}$ is $z$ rotated to the opposite angle. This is the operation that
makes Fourier spectra of real-valued signals symmetric, a fact the later
notebooks in this topic rely on.

In [ ]:
# Cell 04 - Conjugate of z2

display(Math(rf"\overline{{z_2}} = {fmt(z2.conjugate())}"))

---
### The modulus

The modulus $|z| = \sqrt{a^2 + b^2}$ is the distance from the origin
to $z$ in the complex plane - the complex analogue of absolute value.
It is just the Pythagorean theorem applied to the real and imaginary parts.

Python's built-in `abs()` works directly on complex numbers, so no manual
`sqrt` is needed. For $z_3 = 7 - 2i$ the result is $\sqrt{49 + 4} = \sqrt{53}$,
which the cell below prints as $7.2801$.

In a Fourier spectrum the modulus of each coefficient is the **amplitude** of
that frequency component, which is why power spectra are plotted from $|z|$
rather than from the raw complex values.

In [ ]:
# Cell 05 - Modulus of z3

display(
    Math(
        rf"\lvert z_3 \rvert = \sqrt{{{z3.real}^2 + ({z3.imag})^2}}"
        f" = {abs(z3):.4f}"
    )
)

---
### The argument (phase angle)

The argument $\arg z = \theta$ is the angle that $z$ makes with the
positive real axis, measured counterclockwise.
`np.angle` returns $\theta$ in radians in the range $(-\pi, \pi]$.

Together, modulus and argument give the **polar form**:
$z = |z|\,e^{i\theta} = |z|(\cos\theta + i\sin\theta)$.

**Why $\arctan(b/a)$ is not enough.** It is tempting to write
$\theta = \arctan(b/a)$, but that formula throws away information: it only sees
the *ratio* $b/a$, so it cannot tell $-5.9 - 7.5i$ from $5.9 + 7.5i$ and always
returns an angle in $(-\pi/2, \pi/2)$. Our $z_1$ is exactly such a case. It lies
in the third quadrant, so its true argument is $-128.19^\circ$, while
$\arctan(-7.5 / -5.9)$ returns $51.81^\circ$ - the correct line through the
origin, but pointing the wrong way along it, off by exactly $180^\circ$.

The fix is the two-argument arctangent, $\operatorname{atan2}(b, a)$, which
takes the signs of $b$ and $a$ separately and so knows the quadrant. `np.angle`
uses it internally, which is why the cell below reports the correct negative
angle without any manual correction.

In [ ]:
# Cell 06 - Argument (phase angle) of z1 in radians and degrees

theta = np.angle(z1)
display(
    Math(
        rf"\arg z_1 = \theta = {theta:.4f}\:\text{{rad}}"
        rf" = {np.degrees(theta):.4f}^\circ"
    )
)